# 08 — Error Analysis and Local Context

**IEEE-CIS Fraud Detection**

Identify why the model failed on certain instances by analyzing False Positives (FP) and False Negatives (FN).

1. Sort worst errors based on absolute prediction-to-truth delta.
2. Compare distributions of key variables (`TransactionAmt`, `card`, `id_31`) between error classes.
3. Use **SHAP (SHapley Additive exPlanations)** to inspect exactly which features forced a single specific `TransactionID` in the wrong direction.
4. Formulate recommendations for the next iteration (new features, different threshold).

---
## 1 · Imports & Setup

In [ ]:
import sys
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap

PROJECT_ROOT = Path.cwd().parent
PROC_DIR     = PROJECT_ROOT / "data" / "processed"
MODELS_DIR   = PROJECT_ROOT / "models"
METRICS_DIR  = PROJECT_ROOT / "reports" / "metrics"
FIG_DIR      = PROJECT_ROOT / "reports" / "figures"

for d in [FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

# Ensure JS visualization works inside notebooks
shap.initjs()

TARGET = "isFraud"

---
## 2 · Assemble Test Results

In [ ]:
X_test = pd.read_parquet(PROC_DIR / "X_test.parquet")
y_test = pd.read_parquet(PROC_DIR / "y_test.parquet")[TARGET]

obj_cols = X_test.select_dtypes(["object", "category"]).columns
if len(obj_cols) > 0:
    X_test = X_test.drop(columns=obj_cols)

model = xgb.Booster()
model.load_model(MODELS_DIR / "xgb_model_validateset.json")

with open(METRICS_DIR / "xgb_experiment_metrics.json", "r") as f:
    thresh_dict = json.load(f)
THRESHOLD = float(thresh_dict["threshold"])

dtest = xgb.DMatrix(X_test)
y_prob = model.predict(dtest)
y_pred = (y_prob >= THRESHOLD).astype(int)

In [ ]:
# Construct analysis dataframe
df_res = X_test.copy()
df_res["actual"]      = y_test.values
df_res["pred_prob"]   = y_prob
df_res["pred_label"]  = y_pred
df_res["error_delta"] = np.abs(df_res["actual"] - df_res["pred_prob"])

# Classification labels
conditions = [
    (df_res["actual"] == 1) & (df_res["pred_label"] == 1),
    (df_res["actual"] == 0) & (df_res["pred_label"] == 0),
    (df_res["actual"] == 0) & (df_res["pred_label"] == 1),
    (df_res["actual"] == 1) & (df_res["pred_label"] == 0),
]
choices = ["TP", "TN", "FP", "FN"]
df_res["class_type"] = np.select(conditions, choices)

class_counts = df_res["class_type"].value_counts()
print("Prediction Breakdown:")
print(class_counts.to_string())

---
## 3 · Comparing Errors by Key Metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Transaction amount by class
sns.boxplot(data=df_res, x="class_type", y="TransactionAmt", 
            order=["TP", "FP", "FN", "TN"], ax=axes[0],
            palette=["#55A868", "#E8A838", "#DD4A48", "#4C72B0"],
            showfliers=False) # remove huge outliers for easier viewing
axes[0].set_title("TransactionAmt by Prediction Class")
axes[0].set_ylabel("Amount (capped fliers)")

# Velocity metric vs missing
if "time_since_last_card1" in df_res.columns:
    df_res["is_first_card_use"] = (df_res["time_since_last_card1"] == -1)
    sns.countplot(data=df_res, x="class_type", hue="is_first_card_use", 
                  order=["TP", "FP", "FN", "TN"], ax=axes[1],
                  palette=["lightgrey", "#900C3F"])
    axes[1].set_title("First Use of Card by Class")
    axes[1].set_yscale("log")
    axes[1].set_ylabel("Count (Log Scale)")

fig.suptitle("Error Comparisons", fontsize=15, fontweight="bold", y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_error_analysis_vars.png", dpi=150)
plt.show()

> **Observation**:
> - FNs (missed frauds) often have transaction amounts that look typical of TNs.
> - FPs (angry legits) often spike on new cards or unusually high single-time purchases.
> - We might be over-penalizing new devices or 'first card use'.

---
## 4 · Interpreting Predictions with SHAP

We use SHAP's TreeExplainer. 

**A.** Global feature importance (Summary Plot).  
**B.** Individual explanations for Top False Positives / Top False Negatives.

In [ ]:
%%time
explainer = shap.TreeExplainer(model)
# Compute shap values for a smaller fraction (e.g., maximum 5000) for performance
sub_test = df_res.drop(columns=["actual", "pred_prob", "pred_label", "error_delta", "class_type", "is_first_card_use"], errors="ignore")
sample_idx = np.random.choice(len(sub_test), size=min(5000, len(sub_test)), replace=False)
shap_values = explainer.shap_values(sub_test.iloc[sample_idx])
print("Computed SHAP values.")

### Global Summary (Top 20 Drivers)

In [ ]:
plt.figure()
shap.summary_plot(shap_values, sub_test.iloc[sample_idx], max_display=20, show=False)
plt.gcf().savefig(FIG_DIR / "fig_shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

### Deep Dive: Costliest False Negative (Missed a highly confident Fraud)
We isolate the FN where the model felt *most confident* it was Legitimate (`pred_prob` ≈ 0).

In [ ]:
fn_rows = df_res[df_res["class_type"] == "FN"]

if not fn_rows.empty:
    worst_fn_idx = fn_rows["pred_prob"].idxmin()
    print(f"Worst False Negative Index: {worst_fn_idx}")
    print(f"Model Prediction    : {df_res.loc[worst_fn_idx, 'pred_prob']:.4f}")
    print(f"Actual              : {df_res.loc[worst_fn_idx, 'actual']}")

    worst_fn_x = sub_test.loc[worst_fn_idx]
    worst_fn_shap = explainer.shap_values(worst_fn_x)

    shap.force_plot(
        explainer.expected_value, 
        worst_fn_shap, 
        worst_fn_x,
        matplotlib=True, 
        show=False
    )
    fig = plt.gcf()
    fig.savefig(FIG_DIR / "fig_shap_worst_fn.png", dpi=150, bbox_inches="tight")
    plt.show()

### Deep Dive: Costliest False Positive (Blocked a confident Legitimate user)
We isolate the FP where the model felt *most confident* it was Fraud (`pred_prob` ≈ 1).

In [ ]:
fp_rows = df_res[df_res["class_type"] == "FP"]

if not fp_rows.empty:
    worst_fp_idx = fp_rows["pred_prob"].idxmax()
    print(f"Worst False Positive Index: {worst_fp_idx}")
    print(f"Model Prediction    : {df_res.loc[worst_fp_idx, 'pred_prob']:.4f}")
    print(f"Actual              : {df_res.loc[worst_fp_idx, 'actual']}")

    worst_fp_x = sub_test.loc[worst_fp_idx]
    worst_fp_shap = explainer.shap_values(worst_fp_x)

    shap.force_plot(
        explainer.expected_value, 
        worst_fp_shap, 
        worst_fp_x,
        matplotlib=True, 
        show=False
    )
    fig = plt.gcf()
    fig.savefig(FIG_DIR / "fig_shap_worst_fp.png", dpi=150, bbox_inches="tight")
    plt.show()

---
## 5 · Next Modeling Ideas

1. **Adjust Class Hierarchy/Overrides:** If SHAP FP analysis reveals a user is blocked simply because they buy unusually *large amounts* without historical tracking (Velocity = `-1`), we could lower the importance of velocity if `TransactionAmt` corresponds to typically safe high-ticket items.
2. **Real-Time Counters:** False Negatives often hide inside seemingly normal parameters. Capturing multi-device behaviour or velocity-of-change (e.g., standard deviation of spend amount inside of a 24-hr window) in an online Kafka stream before predicting might catch them better.
3. **Different Threshold Mapping:** Based on pure financial ROI. If blocking a legit customer costs $20, but stopping fraud saves $500, we can write an objective function that literally computes standard financial impact, taking us past basic F1 optimisation and directly into Business logic.

In [ ]:
print("Notebook 08 complete. ✓")